# 09 — Generalization Check: FD002 and FD003 (evaluation-only)

**Scope, deliberately lighter than the FD001 build (Phases 1-6):** this
notebook checks whether the pipeline — regime clustering/normalization,
feature engineering, RSF — generalizes to "regimes only" (FD002) and
"fault modes only" (FD003) in isolation, each tested separately before
FD004 combines both. No hyperparameter tuning, no full Cox/Weibull/RSF
ladder, no deployment artifacts — RSF only (the Phase 6 leader), evaluated
once against each subset's official test set. This is the same weight as
the original Phase 1 EDA, not a second full build.

**FD002 (6 operating regimes, 1 fault mode)** needs the regime clustering
and within-cluster normalization built in `features.py` — without it, a
sensor's absolute level shift between regimes would swamp any degradation
signal. **FD003 (1 regime, 2 fault modes)** needs no clustering — it's
structurally identical to FD001, just with a second failure mode mixed
into the same engines, which tests whether the existing FD001-shaped
pipeline handles that without modification.


In [1]:
import sys
from pathlib import Path
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines.utils import concordance_index

sys.path.append(str(Path("..") / "src"))
from turbofan_rul import data_loading, evaluation, features, labeling, models, plotting

plotting.style()
CAP, DURATION_EPS = 125, 1e-3
DATA_DIR = Path("..") / "data" / "raw"
PROCESSED_DIR = Path("..") / "data" / "processed"
SENSOR_COLS = [f"sensor_{i}" for i in range(1, 22)]

## FD002: regime clustering + normalization + sensor selection

In [2]:
data_loading.download_cmapss("FD002", data_dir=DATA_DIR)
fd002 = data_loading.load_subset("FD002", data_dir=DATA_DIR)

kmeans_fd002 = features.fit_regime_clusters(fd002.train, n_clusters=6)
train_regime_labels = features.assign_regime_clusters(fd002.train, kmeans_fd002)
print("cluster sizes:", np.bincount(train_regime_labels))

norm_stats_fd002 = features.fit_regime_normalization_stats(fd002.train, SENSOR_COLS, train_regime_labels)
fd002_train_norm = features.apply_regime_normalization(fd002.train, SENSOR_COLS, train_regime_labels, norm_stats_fd002)

# Why normalization is needed, measured rather than asserted: the share of
# each sensor's raw variance explained by which regime the row is in.
regimes = pd.Series(train_regime_labels, index=fd002.train.index)
def share_explained_by_regime(col):
    x = fd002.train[col]
    within = x.groupby(regimes).transform("mean")
    return 1 - ((x - within) ** 2).sum() / ((x - x.mean()) ** 2).sum()
print(f"sensor_2 raw variance explained by regime: {share_explained_by_regime('sensor_2'):.2%} "
      f"(raw std {fd002.train['sensor_2'].std():.2f} vs. within-regime std "
      f"{fd002.train['sensor_2'].groupby(regimes).std().mean():.2f})")

# Sensor selection must happen AFTER normalization here -- before
# normalization, a sensor's variance is dominated by which regime an
# engine is in, not how informative it is about degradation.
# Caveat: after within-regime z-scoring, any sensor that varies at all within
# its regime has std ~1, so this threshold only drops sensors that are
# constant within every regime -- it doesn't test informativeness.
std_fd002 = fd002_train_norm[SENSOR_COLS].std().sort_values()
display(std_fd002)

STD_THRESHOLD = 0.05
kept_sensors_fd002 = std_fd002[std_fd002 > STD_THRESHOLD].index.tolist()
print(f"\n{len(kept_sensors_fd002)} kept sensors (FD001 comparison: 12 kept, out of the same 21)")

cluster sizes: [13458  8122  8002  8044  8096  8037]
sensor_2 raw variance explained by regime: 99.99% (raw std 37.29 vs. within-regime std 0.44)


sensor_18    0.000000e+00
sensor_5     2.016000e-07
sensor_19    5.058252e-07
sensor_1     9.786754e-06
sensor_16    3.886710e-01
sensor_10    9.220995e-01
sensor_6     9.999499e-01
sensor_15    9.999532e-01
sensor_21    9.999534e-01
sensor_20    9.999534e-01
sensor_8     9.999534e-01
sensor_13    9.999534e-01
sensor_11    9.999535e-01
sensor_12    9.999535e-01
sensor_2     9.999535e-01
sensor_7     9.999535e-01
sensor_17    9.999535e-01
sensor_3     9.999535e-01
sensor_4     9.999535e-01
sensor_14    9.999535e-01
sensor_9     9.999535e-01
dtype: float64


17 kept sensors (FD001 comparison: 12 kept, out of the same 21)


## FD002: fit RSF, evaluate against the official test set

In [3]:
train_survival_fd002 = labeling.build_train_survival_frame(fd002_train_norm, cap=CAP)
train_survival_fd002 = features.build_feature_matrix(train_survival_fd002, kept_sensors=kept_sensors_fd002)
META_COLS = {"unit_number", "time_cycles", "RUL", "duration", "event"}
full_cols_fd002 = [c for c in train_survival_fd002.columns if c not in META_COLS]
print(f"{len(full_cols_fd002)} features, {len(train_survival_fd002)} rows")

# row_stride=15 (much coarser than FD001's 3) purely for notebook runtime --
# FD002 has 2.6x FD001's row count (53,759 vs 20,631) before any subsampling,
# and this notebook is deliberately lighter-weight than a full model build
# (see intro) -- the tradeoff is stated explicitly in the Verdict section below.
t0 = time.time()
rsf_fd002 = models.fit_random_survival_forest(
    train_survival_fd002, full_cols_fd002, row_stride=15, duration_eps=DURATION_EPS, random_state=42
)
print(f"RSF fit: {time.time() - t0:.1f}s")

test_regime_labels = features.assign_regime_clusters(fd002.test, kmeans_fd002)
fd002_test_norm = features.apply_regime_normalization(fd002.test, SENSOR_COLS, test_regime_labels, norm_stats_fd002)
test_featured_fd002 = features.build_feature_matrix(fd002_test_norm, kept_sensors=kept_sensors_fd002)
test_survival_fd002 = labeling.build_test_survival_frame(test_featured_fd002, fd002.rul)

pred_fd002 = models.predict_rul_from_survival(rsf_fd002, test_survival_fd002[full_cols_fd002])
true_fd002 = test_survival_fd002["true_rul"].values

results_fd002 = evaluation.evaluate_rul_predictions(
    true_fd002, pred_fd002["pred_rul"].values, pred_fd002["pred_ci_low"].values,
    pred_fd002["pred_ci_high"].values, model_name="RSF (FD002)",
)
results_fd002["cindex"] = concordance_index(true_fd002, pred_fd002["pred_rul"].values, event_observed=np.ones(len(true_fd002)))
print(results_fd002)

314 features, 53759 rows


RSF fit: 15.4s


{'model': 'RSF (FD002)', 'nasa_score': 10330.21909527645, 'nasa_score_mean': 39.88501581187818, 'late_prediction_rate': 0.4671814671814672, 'rmse': 28.136441678114412, 'mae': 19.45779917712328, 'interval_coverage': 0.7606177606177607, 'cindex': np.float64(0.8575733831595443)}


## FD003: no clustering needed, structurally identical to FD001

In [4]:
data_loading.download_cmapss("FD003", data_dir=DATA_DIR)
fd003 = data_loading.load_subset("FD003", data_dir=DATA_DIR)

cv_fd003 = (fd003.train[SENSOR_COLS].std() / fd003.train[SENSOR_COLS].mean().abs())
kept_sensors_fd003 = cv_fd003[cv_fd003 > 1e-4].index.tolist()
print(f"{len(kept_sensors_fd003)} kept sensors: {kept_sensors_fd003}")

train_survival_fd003 = labeling.build_train_survival_frame(fd003.train, cap=CAP)
train_survival_fd003 = features.build_feature_matrix(train_survival_fd003, kept_sensors=kept_sensors_fd003)
full_cols_fd003 = [c for c in train_survival_fd003.columns if c not in META_COLS]

t0 = time.time()
rsf_fd003 = models.fit_random_survival_forest(
    train_survival_fd003, full_cols_fd003, row_stride=8, duration_eps=DURATION_EPS, random_state=42
)
print(f"RSF fit: {time.time() - t0:.1f}s")

test_featured_fd003 = features.build_feature_matrix(fd003.test, kept_sensors=kept_sensors_fd003)
test_survival_fd003 = labeling.build_test_survival_frame(test_featured_fd003, fd003.rul)
pred_fd003 = models.predict_rul_from_survival(rsf_fd003, test_survival_fd003[full_cols_fd003])
true_fd003 = test_survival_fd003["true_rul"].values

results_fd003 = evaluation.evaluate_rul_predictions(
    true_fd003, pred_fd003["pred_rul"].values, pred_fd003["pred_ci_low"].values,
    pred_fd003["pred_ci_high"].values, model_name="RSF (FD003)",
)
results_fd003["cindex"] = concordance_index(true_fd003, pred_fd003["pred_rul"].values, event_observed=np.ones(len(true_fd003)))
print(results_fd003)

14 kept sensors: ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_6', 'sensor_7', 'sensor_9', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']


RSF fit: 10.8s


{'model': 'RSF (FD003)', 'nasa_score': 818.4227261783789, 'nasa_score_mean': 8.18422726178379, 'late_prediction_rate': 0.6, 'rmse': 17.776751811399198, 'mae': 13.342107074127716, 'interval_coverage': 0.81, 'cindex': np.float64(0.8691398287810844)}


## Generalization summary

In [5]:
# FD001 baseline: Phase 6's RSF on the official FD001 test set, read from the
# saved comparison. test_cindex is engine-level C on point predictions -- the
# same quantity computed for FD002/FD003 below (not the 0.901 row-level
# validation C from notebook 05, which measures something different).
fd001 = pd.read_csv(PROCESSED_DIR / "model_comparison_fd001.csv", index_col="model").loc["Random Survival Forest"]
summary = pd.DataFrame([
    {"subset": "FD001 (baseline, Phase 6)", "regimes": 1, "fault_modes": 1,
     "cindex": fd001["test_cindex"], "nasa_score_mean": fd001["nasa_score"] / 100, "mae": fd001["mae"]},
    {"subset": "FD003 (fault modes only)", "regimes": 1, "fault_modes": 2,
     "cindex": results_fd003["cindex"], "nasa_score_mean": results_fd003["nasa_score_mean"], "mae": results_fd003["mae"]},
    {"subset": "FD002 (regimes only)", "regimes": 6, "fault_modes": 1,
     "cindex": results_fd002["cindex"], "nasa_score_mean": results_fd002["nasa_score_mean"], "mae": results_fd002["mae"]},
]).round(3)
display(summary)

,subset,regimes,fault_modes,cindex,nasa_score_mean,mae
0,"FD001 (baseline, Phase 6)",1,1,0.864,4.625,11.990
1,FD003 (fault modes only),1,2,0.869,8.184,13.342
2,FD002 (regimes only),6,1,0.858,39.885,19.458


## Verdict

All three rows above use the same metrics on each subset's official test
set (engine-level concordance, NASA score per engine, MAE), so they compare
directly.

**Ranking (concordance) generalizes to both subsets.** Test concordance is
essentially the same across FD001 (0.864), FD003 (0.869) and FD002 (0.858)
-- differences of about 0.01, small for test sets of 100-259 engines. RSF
orders engines by remaining life about as well on the harder subsets as on
the baseline.

**Accuracy is where the subsets differ:**
- **FD003 (fault modes only) generalizes well.** NASA score per engine
  (8.2) and MAE (13.3) are somewhat worse than FD001 (4.6 and 12.0), with
  no pipeline changes needed. A second fault mode mixed into
  otherwise-identical engines doesn't meaningfully confuse the feature set.
- **FD002 (regimes only) is a real step down** in point accuracy: NASA
  score per engine 39.9 (about 9x FD001's) and MAE 19.5. This is the honest
  result of one lighter-weight RSF fit, not a full, tuned model -- some of
  the gap likely reflects the coarser `row_stride=15` and the lack of any
  tuning (both adopted for notebook runtime), not only the harder problem.

The clustering step is doing real, verified work -- measured above,
`sensor_2`'s raw variance on FD002 is 99.99% explained by which regime a
row is in (raw std 37.3 vs. 0.44 within a regime) -- but removing that
confound doesn't make FD002 as easy as FD001, and this notebook doesn't
claim it does.

**What this justifies:** FD004 (both axes combined) is the right next
target for the full model ladder and real evaluation -- it's strictly
harder than either subset tested here, so it's the subset that actually
earns the word "generalization," not a formality to complete after two
easy wins.